### Stage 1 — Simple batch: land a bronze table
Ingestion = fetch → flatten → **append** to a bronze table in Unity Catalog.
Bronze keeps raw snapshots: every run appends the full 7-day window with a `fetched_at` timestamp
(the same hour will appear in many snapshots with different values as forecasts get revised).
Reconciliation — latest value per hour — is a silver-layer job later.

In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import functions as F


city_dim_table = "workspace.open_meteo.dim_city"
bronze_table   = "workspace.open_meteo.bronze_hourly_forecast"

rows = spark.sql(f"""
  SELECT latitude, longitude
  FROM {city_dim_table}
  ORDER BY city_id
""").collect()

latitudes = ",".join(str(r.latitude) for r in rows)
longitudes = ",".join(str(r.longitude) for r in rows)

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude":  latitudes,
        "longitude": longitudes,
        "hourly": "temperature_2m,precipitation",
    },
    timeout=30,
)
response.raise_for_status()

spark.createDataFrame([(response.text,)], "payload STRING") \
     .createOrReplaceTempView("open_meteo_raw")

spark.sql(f"""
  SELECT row_number() OVER (ORDER BY city_id) - 1 AS pos, city_id
  FROM {city_dim_table}
""").createOrReplaceTempView("city_order")

# DDL schema of ONE location object in the batched API response (response = ARRAY of these)
RESPONSE_DDL = """
ARRAY<
    STRUCT<
        latitude: DOUBLE,
        longitude: DOUBLE,
        elevation: DOUBLE,
        hourly: STRUCT<
            time: ARRAY<TIMESTAMP>,
            temperature_2m: ARRAY<DOUBLE>,
            precipitation: ARRAY<DOUBLE>
        >
    >
>
"""

spark.sql(f"""
    with parsed as (
        select posexplode(from_json(payload, '{RESPONSE_DDL.strip()}')) as (pos, loc)
        from open_meteo_raw
    )
    select
        o.city_id,
        loc.latitude, 
        loc.longitude, 
        loc.elevation,
        h.time, 
        h.temperature_2m, 
        h.precipitation
    from parsed
    join city_order o on o.pos = parsed.pos
    cross join lateral inline_outer(
        arrays_zip(loc.hourly.time, loc.hourly.temperature_2m, loc.hourly.precipitation)
    ) as h (time, temperature_2m, precipitation)
""").withColumn(
    "fetched_at", F.lit(datetime.now(timezone.utc))
).write.mode("append").saveAsTable(bronze_table)